# The Friendly Fraudsters | NatWest: Card Dispute Agent

FSI Hackathon London, 25 November 2026

## 1. The problem

UK banks handle thousands of card disputes: *"I didn't make this payment"*, *"I cancelled and they still charged me"*, *"my sofa never arrived"*. For each one, a disputes analyst works out what kind of problem it is, looks up the payment and the customer's history, decides who is right, writes an evidence pack for the chargeback and writes to the customer. It is slow, repetitive work, and vulnerable customers need extra care under the FCA's Consumer Duty.

**Who it helps:** the bank's disputes operations team, and the customers waiting for an answer.

**What we built:** a LangGraph team of agents that checks vulnerability, classifies the dispute, routes it to a fraud, merchant or friendly-fraud specialist, drafts the evidence pack, has a quality reviewer check it, pauses for a human to approve, writes the customer letter and remembers the outcome.

## 2. Setup

Run this notebook from the `the_friendly_fraudsters/` folder, with the repo's `.env` filled in (see the README). All data is synthetic.

In [1]:
import json
from collections import Counter
from IPython.display import Markdown, display

from utils import db

complaints = db.table("complaints")
print(f"{len(complaints)} complaints, {len(db.table('customers'))} customers, "
      f"{len(db.table('transactions'))} transactions, {len(db.table('dispute_history'))} past disputes")

25 complaints, 17 customers, 39 transactions, 5 past disputes


## 3. The dataset

Every complaint is labelled with the correct reason code, route, recommendation and vulnerability level, plus a one-line note on why. The cases cover every path through the agent, including traps: a vague "I don't recognise this", an angry customer who is *not* vulnerable, and a crisis case that must go straight to a human.

In [2]:
rows = ["| Case | Customer says | Reason code | Route | Vulnerability |", "| --- | --- | --- | --- | --- |"]
for c in complaints:
    text = c["text"] if len(c["text"]) < 90 else c["text"][:87] + "..."
    rows.append(f"| {c['complaint_id']} | {text} | {c['expected_reason_code']} | {c['expected_route']} | {c['expected_vulnerability']['level']} |")
display(Markdown("\n".join(rows)))

| Case | Customer says | Reason code | Route | Vulnerability |
| --- | --- | --- | --- | --- |
| CMP-001 | I didn't make this payment to GadgetHub. I was asleep at 3am and I've never shopped the... | fraud_card_not_present | fraud | none |
| CMP-002 | I cancelled StreamMax last month but they charged me again. | cancelled_recurring | merchant | none |
| CMP-003 | I paid for a sofa three weeks ago and it never arrived. The shop won't answer my calls ... | goods_not_received | merchant | medium |
| CMP-004 | This PixelGames charge isn't me, I never bought anything. | fraud_card_not_present | friendly_fraud | none |
| CMP-005 | My wallet was stolen on Saturday afternoon in central London and someone used my card a... | fraud_card_present | fraud | none |
| CMP-006 | I don't recognise this payment to PAYPAL *DIGIMARKT. What is it? | fraud_card_not_present | fraud | none |
| CMP-007 | RailGo took the money for my train ticket twice on the same morning. | duplicate_charge | merchant | none |
| CMP-008 | Bella Cucina charged me £148 but our bill was £48. I think they added an extra 1 by mis... | incorrect_amount | merchant | none |
| CMP-009 | I never ordered from FoodDash on the 20th, this isn't mine. | fraud_card_not_present | friendly_fraud | none |
| CMP-010 | I'm 82 and I don't really understand online payments, my grandson set it up for me. A m... | fraud_card_not_present | fraud | medium |
| CMP-011 | Someone took £600 from my account last night and now I can't pay my rent. I can't cope ... | fraud_card_not_present | fast_lane | high |
| CMP-012 | I'm absolutely furious! I cancelled StreamMax in October and they've charged me AGAIN. ... | cancelled_recurring | merchant | none |
| CMP-013 | Something's wrong with my statement, there's a payment on there I don't understand. | fraud_card_not_present | friendly_fraud | none |
| CMP-014 | I want my money back for the FlyHigh charge. I never got to use it. | goods_not_received | merchant | none |
| CMP-015 | Can you look at the £45 from GymPlus? I thought I'd stopped that. | cancelled_recurring | merchant | none |
| CMP-016 | My blender from KitchenPro arrived smashed, and on top of that I think they charged me ... | not_as_described | merchant | none |
| CMP-017 | StyleLane's delivery was three days late, which was annoying, but the real problem is t... | incorrect_amount | merchant | none |
| CMP-018 | I cancelled MealBox ages ago, and now there's a £54 charge from them at 4am that I defi... | fraud_card_not_present | fraud | none |
| CMP-019 | There's a £149.99 GameVault payment from the middle of the night. That wasn't me. | fraud_card_not_present | fraud | none |
| CMP-020 | Someone has used my card at ToyWorld! I never shop there. | fraud_card_not_present | friendly_fraud | none |
| CMP-021 | Money's a bit tight this month so I really need this £60 back. QuickFix Repairs never t... | goods_not_received | merchant | low |
| CMP-022 | I've just moved house and I'm still sorting everything out, but I don't think I ordered... | fraud_card_not_present | friendly_fraud | low |
| CMP-023 | Sorry if this is confusing, I've got a new baby and haven't slept. I think BabyCo charg... | duplicate_charge | merchant | low |
| CMP-024 | This is a bloody joke. Third time I've had to write in. PhoneHub charged me £95 when th... | incorrect_amount | merchant | none |
| CMP-025 | After everything this year I feel like there's no way out. Now £400 has gone to somethi... | fraud_card_not_present | fast_lane | high |

In [3]:
print("Routes:", dict(Counter(c["expected_route"] for c in complaints)))
print("Vulnerability:", dict(Counter(c["expected_vulnerability"]["level"] for c in complaints)))
print("Reason codes:", dict(Counter(c["expected_reason_code"] for c in complaints)))

Routes: {'fraud': 6, 'merchant': 12, 'friendly_fraud': 5, 'fast_lane': 2}
Vulnerability: {'none': 18, 'medium': 2, 'high': 2, 'low': 3}
Reason codes: {'fraud_card_not_present': 12, 'cancelled_recurring': 3, 'goods_not_received': 3, 'fraud_card_present': 1, 'duplicate_charge': 2, 'incorrect_amount': 3, 'not_as_described': 1}


### Data checks

Every complaint must point to a real customer and payment, and the triage rules must agree with every route label.

In [4]:
from evals.check_data import check
problems = check()
print("\n".join(problems) if problems else "All good: every case fits together.")

  Known triage gap (expected, for Pradeep to fix): CMP-019 rules give 'friendly_fraud', truth is 'fraud'
All good: every case fits together.


### Test sets for the quality reviewer and the letters

- **Quality reviewer:** 8 evidence packs, 2 good and 6 with one planted mistake (a made-up fact, a recommendation that contradicts the evidence, a missing section, ignored vulnerability, an accusing tone, a wrong reason code).
- **Customer letters:** 8 scenarios with rules for what each letter must and must not say, for example a decline must mention the Financial Ombudsman Service, and a friendly-fraud letter must never accuse.

In [5]:
from evals.test_sets import qa_packs, letter_cases
for p in qa_packs():
    print(f"{p['pack_id']}  {p['complaint_id']}  expected: {p['expected_verdict']:<4}  planted issue: {p['expected_issue']}")
print()
for l in letter_cases():
    print(f"{l['letter_id']}  {l['complaint_id']}  {l['outcome']:<26} vulnerable={l['vulnerable']}")

QA-01  CMP-003  expected: pass  planted issue: none
QA-02  CMP-001  expected: pass  planted issue: none
QA-03  CMP-001  expected: fail  planted issue: made_up_fact
QA-04  CMP-004  expected: fail  planted issue: recommendation_mismatch
QA-05  CMP-007  expected: fail  planted issue: missing_section
QA-06  CMP-010  expected: fail  planted issue: vulnerability_ignored
QA-07  CMP-009  expected: fail  planted issue: accusing_tone
QA-08  CMP-002  expected: fail  planted issue: wrong_reason_code

LTR-01  CMP-001  file_chargeback            vulnerable=False
LTR-02  CMP-003  file_chargeback            vulnerable=True
LTR-03  CMP-004  decline_claim              vulnerable=False
LTR-04  CMP-009  decline_claim              vulnerable=False
LTR-05  CMP-010  file_chargeback            vulnerable=True
LTR-06  CMP-002  needs_more_info            vulnerable=False
LTR-07  CMP-012  file_chargeback            vulnerable=False
LTR-08  CMP-011  fast_lane_holding_message  vulnerable=True


## 4. One case end to end

**CMP-003:** Amara paid £820 for a sofa that never arrived. She says she is "really struggling", and her profile is marked vulnerable. We run the real agent and watch every step, including the quality reviewer and the pause for a human.

In [6]:
import time
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from agent import build_agent

app = build_agent(checkpointer=InMemorySaver())   # the checkpointer lets the graph pause for the human and resume
config = {"configurable": {"thread_id": "demo-sofa"}}

start = time.perf_counter()
steps = []
for update in app.stream({"complaint_id": "CMP-003"}, config, stream_mode="updates"):
    for step, values in update.items():
        steps.append(step)
        if step == "classify_dispute":
            print(f"classify_dispute   -> reason code: {values['reason_code']}, vulnerability signals: {values['vulnerable_signals']}")
        elif step == "triage":
            print(f"triage             -> route: {values['route']}")
        elif step.endswith("_specialist"):
            print(f"{step:<18} -> recommends: {values['recommendation']}")
        elif step == "quality_review":
            print(f"quality_review     -> {values['review_result']}, next: {values['send_back_to']}")
        elif step == "__interrupt__":
            print("human_approval     -> PAUSED, waiting for a member of staff")
        else:
            print(step)

load_case


classify_dispute   -> reason code: goods_not_received, vulnerability signals: ["I'm really struggling."]
triage             -> route: merchant


merchant_specialist -> recommends: file_chargeback


draft_evidence_pack


quality_review     -> pass, next: human_approval
human_approval     -> PAUSED, waiting for a member of staff


### The evidence pack the human sees

This is what the disputes analyst reads before approving. It has already passed the quality reviewer.

In [7]:
state = app.get_state(config).values
display(Markdown(state["evidence_pack"]))

# Chargeback Evidence Pack

## Summary
The customer reports that a sofa purchased from HomeStyle Furniture has not arrived and that the shop is not answering her calls. The dispute was opened on 25 November 2026. The available facts support filing under **goods not received**.

## Reason code
**Goods not received** — routed as a merchant dispute.

## Transaction details
- **Transaction ID:** TX-5005
- **Customer:** Amara Okafor (CUST-103)
- **Merchant:** HomeStyle Furniture
- **Purchase date and time:** 5 November 2026, 10:20
- **Amount:** £820.00
- **Category:** Furniture
- **Country:** GB
- **Channel:** Online
- **Device known:** Yes
- **3DS:** Passed
- **Dispute opened:** 25 November 2026

## Evidence
- The customer states: “I paid for a sofa three weeks ago and it never arrived.”
- The customer reports that the shop will not answer her calls.
- The recorded dispute outcome is **pending**; the notes say to approve filing a chargeback.
- The available facts support proceeding under the goods-not-received reason code.

## Customer history
- Customer since 2018.
- No earlier purchase history with this merchant is recorded.

## Vulnerability
The customer is recorded as vulnerable. The complaint includes: “I’m really struggling.” No further vulnerability details are provided.

## Recommendation
**File chargeback** under the goods-not-received reason code.

## Points for the reviewer
- Confirm whether any delivery or non-delivery documentation is available; none is described in the supplied facts.
- The customer says the purchase was three weeks ago, while the transaction and dispute dates are 5 and 25 November respectively (20 days apart). Check whether this difference needs clarification.
- The customer’s report that the shop is not answering calls is not accompanied by call records or other supporting documentation.
- The dispute outcome remains pending; the notes recommend approving filing.

### The pause for a human

The graph stops at `human_approval` and hands the reviewer everything they need. The AI never has the final say.

In [8]:
pause = app.get_state(config).tasks[0].interrupts[0].value
print("Recommendation:", pause["recommendation"])
print("Vulnerable customer:", pause["vulnerable_customer"])
print("Quality review:", pause["quality_review"])

Recommendation: file_chargeback
Vulnerable customer: True
Quality review: {'result': 'pass', 'retries': 0, 'outstanding_issues': ''}


### The analyst approves, and the customer letter is written

In [9]:
final = app.invoke(Command(resume={"decision": "approve", "notes": "Evidence supports non-delivery."}), config)
print(f"Total time end to end: {time.perf_counter() - start:.0f} seconds (including the human step in this notebook)")
print(f"Evidence pack rewrites requested by the reviewer: {final.get('retry_count', 0)}")
print()
print(final["customer_letter"])

Total time end to end: 25 seconds (including the human step in this notebook)
Evidence pack rewrites requested by the reviewer: 0

Subject: Update on your sofa payment dispute

Dear Amara Okafor,

We have reviewed your dispute about the £820.00 payment to HomeStyle Furniture on 5 November 2026. Your sofa has not arrived.

We are asking HomeStyle Furniture to return the £820.00 on your behalf. The money has not been returned yet. HomeStyle Furniture has up to 45 days to respond.

We will write to you as soon as we hear back, or within 45 days at the latest.

There is no pressure or rush to talk with us. If you would like to talk this through, call our support team free on 0800 000 0000. They are available from 8am to 8pm, 7 days a week. You can ask for more time or for someone to help you at any point.

Yours sincerely,
Card Disputes Team
Friendly Bank


### Does the letter follow the rules?

The letter for a vulnerable customer must give the amount, a support contact and next steps, with no pressure.

In [10]:
from evals.test_sets import letter_cases, check_letter
rules = next(c for c in letter_cases() if c["letter_id"] == "LTR-02")
problems = check_letter(final["customer_letter"], rules)
print("\n".join(problems) if problems else "All letter rules pass.")

All letter rules pass.


## 5. Evals

**Baseline (before the new agents), 25 cases** (LangSmith dataset `friendly-fraudsters-disputes-v3`):

| Measure | Result |
| --- | --- |
| Reason code correct | 23 / 25 |
| Route correct | 21 / 25 |
| Vulnerability flagged correctly | 24 / 25 |
| Recommendation correct (fast-lane cases excluded) | 19 / 23 |
| High-risk cases sent to the fast lane | **0 / 2** |
| Median time per case (up to the specialist) | 3.6 s |

**Every miss explained:**
- **CMP-011, CMP-025 (crisis cases):** the agent noticed the vulnerability but had no fast lane yet, so both went through normal fraud handling. *Fix: vulnerability_check + fast lane.*
- **CMP-016 (mixed problem):** picked "duplicate charge" because the customer said "I think they charged me twice", but the data shows one payment. *Fix: classifier should check the claim against the payments.*
- **CMP-018 (mixed problem):** picked "cancelled subscription" from "I cancelled ages ago", missing the 4am foreign payment on an unknown device. *Fix: weigh the payment evidence over the customer's framing.*
- **CMP-019 (lookalike):** triage rules called real fraud "friendly fraud" because of an old purchase and a past lost claim, ignoring the unknown device and failed 3-D Secure. *Fix: an unknown device + failed 3-D Secure should block the friendly-fraud route.*
- **CMP-014, 015, 016, 024 (recommendation):** the merchant specialist declined all four because the data holds no proof (no cancellation email, no delivery record). In a real chargeback the customer's credible account is enough to raise it, and the merchant must prove otherwise. *Fix: decline only when the evidence contradicts the customer.*
- **CMP-022 (low vulnerability):** missed "I've just moved house" as a life event. *Fix: dedicated vulnerability_check prompt.*

## 6. Results table

*To add.*

## 7. KPIs and ROI

**Public figures (all verified from the source pages, no internal data):**

| Figure | Value | Source |
| --- | --- | --- |
| UK remote purchase (card-not-present) fraud cases, 2024 | nearly **2.6 million** (up 22%), losses just under £400M | [UK Finance, Annual Fraud Report 2025 press release](https://www.ukfinance.org.uk/news-and-insight/press-release/fraud-report-2025-press-release) |
| UK confirmed unauthorised fraud cases, 2024 | **3.13 million**, £722M lost | UK Finance, as above |
| Complaints to UK firms, 2025 H2 | **1.74M** total; credit cards 218,456; current accounts 492,149; **55.5% upheld** | [FCA aggregate complaints data 2025 H2](https://www.fca.org.uk/data/aggregate-complaints-data-2025-h2) |
| NatWest share of UK current accounts | **16.5%**, 19M Retail Banking customers (Dec 2025) | [NatWest Group investment case](https://investors.natwestgroup.com/our-investment-case) |
| Cost for a bank to process one dispute | **$9.08–$10.32** (US benchmark) ≈ £7.20–£8.15 at $1 ≈ £0.79 | [Mastercard, true cost of a chargeback](https://www.mastercard.com/gb/en/news-and-trends/Insights/2025/what-s-the-true-cost-of-a-chargeback-in-2025.html) |
| Consumers who have mistakenly disputed a real purchase | **48%** (supports our friendly-fraud design) | Mastercard, as above |

**ROI estimate (fraud disputes only, so conservative):**

| Step | Figure | Type |
| --- | --- | --- |
| UK card-not-present fraud cases a year | 2.6M | Sourced |
| NatWest's share, if fraud is proportional to current-account share | 16.5% → about **430,000 cases a year** | Assumption (sourced share) |
| Cost per dispute | about **£7.70** | Sourced (US benchmark) |
| Handling effort removed by the agent (a human still reviews every case) | **60%** | Assumption, team to agree |
| AI cost per case | measured from LangSmith traces | Measured |
| **Estimated saving** | 430,000 × £7.70 × 60% ≈ **£2M a year**, minus AI cost | |

Caveats to say out loud: the cost figure is a US benchmark; merchant disputes and friendly fraud are **not** counted, so the real figure is likely higher; UK Finance published 2025 figures on 25 Nov 2026 that we have not yet checked. Beyond money: faster answers, consistent vulnerable-customer care, and no missed 8-week deadlines.

## 8. Next steps

*To add.*